# Examine Relationship Between Combine Results And WR Separation During Hitch/In/Out Routes

**<span style="color: gold;">PROJECT</span>**

WR/DB Separation Analysis

**<span style="color: gold;">GOAL OF THIS NOTEBOOK</span>**

We have previously shown that some of our engineered features derived from NFL Combine player-tracking data have significant associations with a WRs ability to create separation on cutting routes.

We now want to test if that information tells us more than what general Combine results (40-yard dash time, 3 Cone drill time, etc.) tell us. That is, if coaches could learn just as much from simply looking at general Combine results, then they don't need our derived metrics. 

Therefore, we will test here if there are any significant relationships between general Combine metrics and a WRs ability to create space during cuts in hitch/in/out routes. We hypothesize that no Combine features will have any predictive power on a WRs ability to create space.

Predictors: General Combine Results

Outcome: Separation At Cut During Hitch/In/Out Routes

**<span style="color: gold;">FINDINGS</span>**

No meaningful relationships between Combine results and separation created on cutting routes.

**<span style="color: gold;">NOTES</span>**

Since we're not using the separation_at_pass_forward variable from the BDB2027 data, but rather creating separation outcome variables from player-tracking data, we are only using data from *2023 Rookie WRs*.

<br><br/>
#### *Initialize Environment*

In [ ]:
import sys
from pathlib import Path

# Identify Project Root
PROJECT_ROOT = next(
    parent
    for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / "pyproject.toml").exists()
)
sys.path.insert(0, str(PROJECT_ROOT))

# Import Shared Project Paths
from utils.paths import (
    CLEAN_DATA_DIR,
    SEPARATION_DATA_DIR,
)

import polars as pl 
import pandas as pd 
import numpy as np
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "png"
from datetime import datetime

import statsmodels.api as sm

# Import schema_map For 2027 BDB Data
import sys
from pathlib import Path
# Identify Project Root

PROJECT_ROOT = next(

    parent

    for parent in [Path.cwd(), *Path.cwd().parents]

    if (parent / "pyproject.toml").exists()

)



# Import Shared Data Schemas

sys.path.insert(0, str(PROJECT_ROOT))

from utils.schema import schema_map

<br><br/>
## Load Data
***

#### Set Parameters For Analysis

In [ ]:
# =======================================
# Import Parameters That Define What 
# Data We'll Examine In This Project
# =======================================
param_path = PROJECT_ROOT / "05_WR_Separation_Analysis" / "Data_Pull"
sys.path.append(str(param_path))
import Data_Parameters as params

In [ ]:
# =======================================
# Specify Routes of Interest
# =======================================

ROUTES_OF_INTEREST = ["HITCH", "IN", "OUT"]

<br><br/>
#### Play and Player Level Details

Data Is From *01_BDB26_WR_Player_Tracking.ipynb*

In [ ]:
# ===========================================
# Player-Time Level WR/DB Player-Tracking & Play Data
# ===========================================

play_player_details_df = pl.read_csv(str(SEPARATION_DATA_DIR / "00_2023_Rookie_Player_Time_Level_Play_Details_Data.csv"))

print("Play/Player Details Data:")
print(f"    - Number of Unique WRs: {play_player_details_df.filter(pl.col("player_position") == "WR").select("nfl_id").n_unique()}")
print(f"    - Number of WR Player-Plays: {play_player_details_df.filter(pl.col("player_position") == "WR").select("player_play_id").n_unique()}")
play_player_details_df

<br><br/>
#### WR Separation Data 

Data Is From *01_BDB26_WR_Player_Tracking.ipynb*

In [ ]:
# =========================================
# WR Separation Data Among 2023 Rookies
# =========================================

wr_separation_df = pl.read_csv(str(SEPARATION_DATA_DIR / "01_2023_Rookie_Player_Time_Level_Separation_Data.csv"))

# -----------------------------------------
# Identify Players Of Interest
# -----------------------------------------

players_of_interest = (
    wr_separation_df
    .select("wr_nfl_id")
    .unique()
    .to_series()
    .to_list()
)

print("Separation Data:")
print(f"    - Dimensions: {wr_separation_df.shape}")
print(f"    - Number of Rookie WRs: {wr_separation_df.select("wr_nfl_id").n_unique()}")
print(f"    - Number of Player-Plays: {wr_separation_df.select("player_play_id").n_unique()}")
wr_separation_df

<br><br/>
#### Combine Results Data

In [ ]:
# =======================================
# Combine Results Data
# =======================================

combine_results_df = (
    pl.read_csv(
        str(CLEAN_DATA_DIR / "combine_results.csv"), 
        schema_overrides = schema_map["combine_results"]
    )
)

combine_features_df = (
    pl.read_csv(str(SEPARATION_DATA_DIR / "03_WR_Combine_Features_Data.csv"))
    .select("nfl_id", "decel_score")
)

combine_results_df = (
    combine_results_df
    .join(
        combine_features_df, 
        on = "nfl_id", 
        how = "left"
    )
)

print("Combine Results Data:")
print(f"    - Dimensions: {combine_results_df.shape}")
print(f"    - Number of Players: {combine_results_df.select("nfl_id").n_unique()}")
combine_results_df

<br><br/>
## Data Cleaning
***

#### Filter Data Based On Inclusion Criteria

In [ ]:
# ===============================
# Number of WRs In WR Separation Data
# ===============================
wr_player_plays = wr_separation_df.select("player_play_id").unique().to_series().to_list()
print("WRs In WR Separation Data:")
print(f"    - Number of Unique WR Player-Plays: {len(wr_player_plays)}")

In [ ]:
# ===============================
# How Many of Those Player-Plays Do We Have Route Data For?
# ===============================

# -------------------------------
# For BDB2027 Route Data, Use route_ran
# -------------------------------

bdb2027_extra_route_data = (
    play_player_details_df
    .filter( 
        pl.col("route_ran").is_not_null()
    )
    .group_by("player_play_id")
    .agg(
        route_ran = pl.col("route_ran").first()
    )
)

# -------------------------------
# For BDB2026 Route Data For Targeted 
# Receivers, Use route_of_targetered_receiver
# -------------------------------

target_routes = (
    play_player_details_df
    .filter(
        pl.col("target"), 
        pl.col("route_of_targeted_receiver").is_not_null()
    )
    .group_by("player_play_id")
    .agg(
        route_of_targeted_receiver = pl.col("route_of_targeted_receiver").first()
    )
)

# -------------------------------
# Target Data
# -------------------------------

target_data = (
    play_player_details_df
    .select("player_play_id", "target")
    .unique()
)

# -------------------------------
# Add Routes To WR Separation Data
# -------------------------------

wr_separation_df = (
    wr_separation_df
    # Add target data
    .join(
        target_data, 
        on = "player_play_id", 
        how = "inner"
    )
    # Add non-targeted routes
    .join(
        bdb2027_extra_route_data, 
        on = "player_play_id", 
        how = "left"
    )
    # Add targeted routes
    .join(
        target_routes, 
        on = "player_play_id", 
        how = "left"
    )
    # Create route variable
    .with_columns(
        route = (
            # If target and has route_of_targeted_receiver data, then use that
            pl.when((pl.col("target") == True) & (pl.col("route_of_targeted_receiver").is_not_null()))
              .then(pl.col("route_of_targeted_receiver"))
            # Otherwise, use route_ran
              .otherwise(pl.col("route_ran"))
        )
    )
)

# -----------------------------
# Get Rid Of Player-Plays With No Route Data
# -----------------------------

wr_separation_df = (
    wr_separation_df
    .filter(pl.col("route").is_not_null())
)

# -----------------------------
# Only Consider Targeted Receivers
# -----------------------------

wr_separation_df = (
    wr_separation_df.filter("target")
)

# -----------------------------
# Only Consider These Route Types
# -----------------------------

wr_separation_df = (
    wr_separation_df
    .filter(pl.col("route").is_in(ROUTES_OF_INTEREST))
)


print(f"Updated Assignments Data:")
print(f"    - Number of WR Player-Plays: {wr_separation_df.select("player_play_id").n_unique()}")
print(f"    - Number of WRs: {wr_separation_df.select("wr_nfl_id").n_unique()}")
print(f"    - Number of Unique Plays: {wr_separation_df.select("game_play_id").n_unique()}")
wr_separation_df

<br><br/>
#### Get Data On Cuts And Separation At Cut From Targeted Routes

In [ ]:
# ==============================================================================
# Extract In-Game Cut Frame and Separation Delta for Routes
# ==============================================================================

# ------------------------------------------------------------------------------
# 1. Calculate Downfield Velocity, Speed, and Linear Acceleration
# ------------------------------------------------------------------------------
kinematics_df = (
    wr_separation_df
    .sort(["player_play_id", "frame_id"])
    .with_columns(
        # Calculate raw directional velocities (dt = 0.1s at 10Hz)
        vx = (pl.col("wr_x") - pl.col("wr_x").shift(1).over("player_play_id")) / 0.1,
        vy = (pl.col("wr_y") - pl.col("wr_y").shift(1).over("player_play_id")) / 0.1,
    )
    .with_columns(
        # Speed is the magnitude of the velocity vector
        s = (pl.col("vx")**2 + pl.col("vy")**2).sqrt()
    )
    .with_columns(
        # Calculate true linear acceleration (including negative braking numbers)
        linear_accel = (pl.col("s") - pl.col("s").shift(1).over("player_play_id")) / 0.1
    )
    .with_columns(
        # Isolate true downfield velocity based on play direction
        v_downfield = pl.when(pl.col("play_direction") == "right")
                      .then(pl.col("vx"))
                      .otherwise(-pl.col("vx"))
    )
)

# ------------------------------------------------------------------------------
# 2. Isolate the Cut Apex Frame Per Route
# ------------------------------------------------------------------------------
cut_apexes_df = (
    kinematics_df
    .filter(
        (pl.col("linear_accel").is_not_null()) & 
        (pl.col("v_downfield") > 4.0)
    )
    .group_by("player_play_id")
    .agg([
        # Find the frame where they slammed on the brakes the hardest to make the cut
        pl.col("frame_id").filter(pl.col("linear_accel") == pl.col("linear_accel").min()).first().alias("cut_frame_id")
    ])
)

# ------------------------------------------------------------------------------
# 3. Extract Separation at Cut Apex and Calculate Separation Creation Delta
# ------------------------------------------------------------------------------
final_in_game_separation_df = (
    kinematics_df
    .join(cut_apexes_df, on="player_play_id", how="inner")
    # Grab the metrics specifically at the cut apex or right after it
    .filter(
        (pl.col("frame_id") == pl.col("cut_frame_id")) | 
        (pl.col("frame_id") == (pl.col("cut_frame_id") + 3))
    )
    .group_by("player_play_id")
    .agg([
        pl.col("wr_nfl_id").first(),
        pl.col("wr_name").first(),
        # Separation exactly at the moment they plant their foot
        pl.col("wr_cb_distance").filter(pl.col("frame_id") == pl.col("cut_frame_id")).first().alias("separation_at_apex"),
        # Separation 0.3 seconds after the cut as they execute the break stem
        pl.col("wr_cb_distance").filter(pl.col("frame_id") == (pl.col("cut_frame_id") + 3)).first().alias("separation_post_cut")
    ])
    .with_columns(
        # Positive values = WR gained separation. Negative values = Defender closed in.
        separation_creation_delta = pl.col("separation_post_cut") - pl.col("separation_at_apex")
    )
)

# ------------------------------------------------------------------------------
# 4. Roll up to ONE row per player to get their average separation metrics
# ------------------------------------------------------------------------------
separation_at_cut_df = (
    final_in_game_separation_df
    .group_by("wr_nfl_id")
    .agg([
        pl.col("wr_name").first(),
        pl.col("separation_at_apex").mean().alias("avg_in_game_separation_at_apex"),
        pl.col("separation_post_cut").mean().alias("avg_in_game_separation_post_cut"),
        pl.col("separation_creation_delta").mean().alias("avg_separation_creation_delta")
    ])
)

separation_at_cut_df


<br><br/>
#### Add Combine Metrics To Player Separation Data

In [ ]:
# ===================================
# Add Combine Features Data
# ===================================
analysis_df = (
    separation_at_cut_df
    .join(
        combine_results_df, 
        left_on = ["wr_nfl_id"], 
        right_on = ["nfl_id"], 
        how = "inner"
    )
)

print("Final Analysis Dataset:")
print(f"    - Number of WRs: {analysis_df.select("wr_nfl_id").n_unique()}")
analysis_df

<br><br/>
## Examine Associations
***

We've previously shown that there is a statistically significant association between *decel_score* and *avg_in_game_separation_post_cut*. We will now see if we can find a statistically significant relationship between *avg_in_game_separation_post_cut* and our general Combine results.

<br><br/>
#### General Combine Results vs Separation Post-Cut

***Scatter Plot***

In [ ]:
predictor = "short_shuttle" 
outcome = "avg_in_game_separation_post_cut"

fig = px.scatter(
    data_frame = analysis_df, 
    x = predictor, 
    y = outcome, 
    trendline = "ols"
)

fig.show()

***Univariate Model***

In [ ]:
# ==========================================
# Univariate Model
# ==========================================

predictor = "short_shuttle" 
outcome = "avg_in_game_separation_post_cut"

regression_df = analysis_df

regression_df = regression_df.filter(
    (pl.col(predictor).is_not_null())
    & 
    (pl.col(outcome).is_not_null())

)

pd_reg = regression_df.to_pandas()

X = pd_reg[predictor]
y = pd_reg[outcome]

X = sm.add_constant(X)

model = sm.OLS(y, X).fit()
print(model.summary())

<br><br/>
## Deceleration vs NGS Athleticism
***

I want to check if our `decel_score` metric is strongly correlated with `ngs_athleticism_score`. I'm checking because I want to know if our new metric adds new and valuable information that coaches wouldn't have been able to know otherwise. 

<br><br/>
#### Examine Correlation

We see below that the correlation between `decel_score` and `ngs_athleticism_score` is low. Since they are not highly correlated, our new metric likely provides new, valuable information.

In [ ]:
# =============================================
# Correlation
# =============================================

print(f"Correlation Between decel_score And ngs_athleticism_score: {round(analysis_df.select(pl.corr("decel_score", "ngs_athleticism_score")).item(), 3)}")

<br><br/>
#### What Does Our Metric Add Beyond `ngs_atleticism_score`?

Does `decel_score` add useful information beyond what is already known by `ngs_athleticism_score`? We will check this by examining how much information is added to a model that includes `ngs_athleticism_score` when `decel_score` is added.

We see below that `decel_score` appears to add valuable information. While `decel_score` has a p-value slight above 0.10, it adds a large amount to R-squared. AIC also decreases a good bit when adding `decel_score`. Also, with such a small sample size, our relatively small p-value for `decel_score`, while adjusting for `ngs_athleticism_score` should not be ignored.

In [ ]:
# ==========================================
# NGS Athleticism Alone
# ==========================================

predictor = "ngs_athleticism_score"

regression_df = analysis_df

regression_df = regression_df.filter(
    (pl.col(predictor).is_not_null())
    & 
    (pl.col(outcome).is_not_null())

)

pd_reg = regression_df.to_pandas()

X = pd_reg[predictor]
y = pd_reg[outcome]

X = sm.add_constant(X)

model = sm.OLS(y, X).fit()
print(model.summary())

In [ ]:
# ==========================================
# Deceleration Score Alone
# ==========================================

predictor = "decel_score"

regression_df = analysis_df

regression_df = regression_df.filter(
    (pl.col(predictor).is_not_null())
    & 
    (pl.col(outcome).is_not_null())

)

pd_reg = regression_df.to_pandas()

X = pd_reg[predictor]
y = pd_reg[outcome]

X = sm.add_constant(X)

model = sm.OLS(y, X).fit()
print(model.summary())

In [ ]:
# ===========================================
# NGS Athleticism Score And Decel Score
# ===========================================

predictors = [
    "ngs_athleticism_score",
    "decel_score", 
]

corr = round(analysis_df.select(pl.corr(predictors[0], predictors[1])).item(), 2)
vif = 1 / (1 - (corr**2))
print(f"Correlation Between {predictors[0]} and {predictors[1]}: {corr} (VIF = {round(vif, 2)})\n")

regression_df = analysis_df.filter(
    pl.all_horizontal(
        [pl.col(col).is_not_null() for col in predictors + [outcome]]
    )
)

pd_reg = regression_df.to_pandas()

X = pd_reg[predictors]
y = pd_reg[outcome]

X = sm.add_constant(X)

model = sm.OLS(y, X).fit()

print(model.summary())